# Privacy-Preserving Federated Network Intrusion Detection System

## Notebook 03 - Data Preprocessing

### Objectives

- Load the official CICIDS2017 dataset
- Merge all dataset files
- Standardize column names
- Clean target labels
- Convert features to numeric values
- Handle missing and infinite values
- Remove duplicate records
- Save a clean unscaled dataset

### Important

This notebook performs data cleaning only.

Feature selection and scaling will be performed after
the train/test split to prevent data leakage.

In [ ]:
# =====================================================
# IMPORT LIBRARIES
# =====================================================

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [ ]:
# =====================================================
# PROJECT CONFIGURATION
# =====================================================

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "CICIDS2017"
)

PROCESSED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Raw data path       : {RAW_DATA_PATH}")
print(f"Processed data path: {PROCESSED_DATA_PATH}")

Raw data path       : c:\MyFiles\Project\Privacy-Preserving-Federated-NIDS\data\raw\CICIDS2017
Processed data path: c:\MyFiles\Project\Privacy-Preserving-Federated-NIDS\data\processed


In [23]:
# =====================================================
# LOAD AND MERGE DATASETS
# =====================================================

csv_files = sorted(
    RAW_DATA_PATH.glob("*.csv")
)

print(f"CSV files found: {len(csv_files)}")

dataframes = []

for file in csv_files:

    print(f"Loading: {file.name}")

    df = pd.read_csv(
        file,
        low_memory=False
    )

    dataframes.append(df)

merged_df = pd.concat(
    dataframes,
    ignore_index=True
)

print("\nDataset merged successfully.")
print(f"Shape: {merged_df.shape}")

Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Wednesday-workingHours.pcap_ISCX.csv

Merged dataset shape: (2830743, 79)


In [24]:
# =====================================================
# CLEAN COLUMN NAMES
# =====================================================

merged_df.columns = (
    merged_df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("/", "_")
    .str.replace("-", "_")
)

print("Column names cleaned.")

print("\nNumber of columns:")
print(len(merged_df.columns))

print("\nFirst 10 columns:")
print(merged_df.columns[:10].tolist())

Column names cleaned.

Number of columns:
79

First 10 columns:
['Destination_Port', 'Flow_Duration', 'Total_Fwd_Packets', 'Total_Backward_Packets', 'Total_Length_of_Fwd_Packets', 'Total_Length_of_Bwd_Packets', 'Fwd_Packet_Length_Max', 'Fwd_Packet_Length_Min', 'Fwd_Packet_Length_Mean', 'Fwd_Packet_Length_Std']


In [25]:
# =====================================================
# IDENTIFY TARGET COLUMN
# =====================================================

label_column = merged_df.columns[-1]

print(f"Target column: {label_column}")

print("\nOriginal class distribution:")
print(
    merged_df[label_column]
    .value_counts()
)

Target column: Label

Original class distribution:
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [26]:
# =====================================================
# CLEAN TARGET LABELS
# =====================================================

merged_df[label_column] = (
    merged_df[label_column]
    .astype(str)
    .str.strip()
)

merged_df = merged_df[
    merged_df[label_column] != ""
]

print("Target labels cleaned.")

print("\nClasses:")
print(
    merged_df[label_column]
    .value_counts()
)

Target labels cleaned.

Classes:
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [27]:
# =====================================================
# SEPARATE FEATURES AND TARGET
# =====================================================

X = merged_df.drop(
    columns=[label_column]
)

y = merged_df[label_column]

print(f"Features: {X.shape}")
print(f"Target  : {y.shape}")

Features: (2830743, 78)
Target  : (2830743,)


In [28]:
# =====================================================
# CONVERT FEATURES TO NUMERIC
# =====================================================

for column in X.columns:

    X[column] = pd.to_numeric(
        X[column],
        errors="coerce"
    )

print("Feature conversion completed.")

Feature conversion completed.


In [29]:
# =====================================================
# HANDLE INFINITE VALUES
# =====================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

print("Infinite values converted to NaN.")

Infinite values converted to NaN.


In [30]:
# =====================================================
# HANDLE MISSING VALUES
# =====================================================

before_nan = len(X)

valid_rows = ~X.isnull().any(axis=1)

X = X.loc[valid_rows].copy()
y = y.loc[valid_rows].copy()

after_nan = len(X)

print(f"Rows before NaN removal: {before_nan:,}")
print(f"Rows after NaN removal : {after_nan:,}")
print(f"Rows removed           : {before_nan - after_nan:,}")

Rows before NaN removal: 2,830,743
Rows after NaN removal : 2,827,876
Rows removed           : 2,867


In [31]:
# =====================================================
# REMOVE DUPLICATE RECORDS
# =====================================================

clean_df = X.copy()

clean_df["Label"] = y

before_duplicates = len(clean_df)

clean_df = clean_df.drop_duplicates()

after_duplicates = len(clean_df)

print(
    f"Rows before duplicate removal: "
    f"{before_duplicates:,}"
)

print(
    f"Rows after duplicate removal : "
    f"{after_duplicates:,}"
)

print(
    f"Duplicates removed           : "
    f"{before_duplicates - after_duplicates:,}"
)

Rows before duplicate removal: 2,827,876
Rows after duplicate removal : 2,520,798
Duplicates removed           : 307,078


In [32]:
# =====================================================
# RESET DATASET INDEX
# =====================================================

clean_df = clean_df.reset_index(
    drop=True
)

print(f"Final clean shape: {clean_df.shape}")

Final clean shape: (2520798, 79)


In [33]:
# =====================================================
# VERIFY CLEAN DATASET
# =====================================================

print("=" * 60)
print("CLEAN DATASET VERIFICATION")
print("=" * 60)

print(f"Rows       : {clean_df.shape[0]:,}")
print(f"Features   : {clean_df.shape[1] - 1}")
print(f"Target     : Label")
print(
    f"Missing    : "
    f"{clean_df.isnull().sum().sum():,}"
)
print(
    f"Duplicates : "
    f"{clean_df.duplicated().sum():,}"
)

print("\nData types:")
print(
    clean_df.dtypes.value_counts()
)

print("\nClass distribution:")
print(
    clean_df["Label"].value_counts()
)

CLEAN DATASET VERIFICATION
Rows       : 2,520,798
Features   : 78
Target     : Label
Missing    : 0
Duplicates : 0

Data types:
int64      54
float64    24
str         1
Name: count, dtype: int64

Class distribution:
Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [34]:
# =====================================================
# SAVE CLEAN UN-SCALED DATASET
# =====================================================

clean_output = (
    PROCESSED_DATA_PATH
    / "CICIDS2017_clean.csv"
)

clean_df.to_csv(
    clean_output,
    index=False
)

print(
    f"Clean dataset saved to:\n"
    f"{clean_output}"
)

print(
    f"\nDataset shape: "
    f"{clean_df.shape}"
)

Clean dataset saved to:
c:\MyFiles\Project\Privacy-Preserving-Federated-NIDS\data\processed\CICIDS2017_clean.csv

Dataset shape: (2520798, 79)


In [ ]:
# =====================================================
# FINAL VALIDATION CHECK A
# CHECK FOR TIMESTAMP / TEMPORAL INFORMATION
# =====================================================

print("=" * 60)
print("CHECKING FOR TEMPORAL INFORMATION")
print("=" * 60)

print("\nDataset shape:")
print(df.shape)

print("\nColumns containing time/date information:")

time_columns = [
    col for col in df.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "timestamp",
            "time",
            "date",
            "day"
        ]
    )
]

if time_columns:

    for col in time_columns:
        print(
            f"FOUND: {col} | "
            f"dtype={df[col].dtype}"
        )

else:

    print(
        "NO timestamp/time/date/day column found."
    )

In [35]:
# =====================================================
# SAVE LABEL MAPPING
# =====================================================

unique_labels = sorted(
    clean_df["Label"].unique()
)

label_mapping = pd.DataFrame({
    "Encoded_Label": range(
        len(unique_labels)
    ),
    "Original_Label": unique_labels
})

label_mapping.to_csv(
    PROCESSED_DATA_PATH
    / "label_mapping.csv",
    index=False
)

print("Label mapping saved.")

label_mapping

Label mapping saved.


,Encoded_Label,Original_Label
0,0,BENIGN
1,1,Bot
2,2,DDoS
3,3,DoS GoldenEye
4,4,DoS Hulk
5,5,DoS Slowhttptest
6,6,DoS slowloris
7,7,FTP-Patator
8,8,Heartbleed
9,9,Infiltration


In [36]:
# =====================================================
# FINAL PREPROCESSING SUMMARY
# =====================================================

print("=" * 60)
print("PREPROCESSING COMPLETED")
print("=" * 60)

print(
    f"Final rows     : "
    f"{clean_df.shape[0]:,}"
)

print(
    f"Final features : "
    f"{clean_df.shape[1] - 1}"
)

print(
    f"Missing values : "
    f"{clean_df.isnull().sum().sum()}"
)

print(
    f"Duplicates     : "
    f"{clean_df.duplicated().sum()}"
)

print(
    "\nOutput:"
)

print(clean_output)

PREPROCESSING COMPLETED
Final rows     : 2,520,798
Final features : 78
Missing values : 0
Duplicates     : 0

Output:
c:\MyFiles\Project\Privacy-Preserving-Federated-NIDS\data\processed\CICIDS2017_clean.csv
